<a href="https://colab.research.google.com/github/arjunnamburi/ml_death_overs_ipl/blob/main/ML_Project_IPL_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
import numpy as np
import pandas as pd

In [31]:
df = pd.read_csv("/content/prog2_renamed.csv")
pd.set_option('display.max_columns', None)

In [32]:
filt = (df['innings'] > 2)
rows_to_drop = df[df['innings'] > 2].index
df.drop(index=rows_to_drop, inplace=True)

In [33]:
inn = (df[df.innings.isin([1, 2])]
       .groupby(['match_id', 'innings']).agg(
            max_over=('over', 'max'),
            runs=('ball_runs_total', 'sum'),
            wkts=('c_team_wickets', 'max'),
            target=('runs_target', 'first')
            ).unstack('innings'))

In [37]:

TARGET_IS_WINNING_SCORE = True   # True if runs_target == innings-1 total + 1 in a normal match
FULL_BALLS = 120                 # legal balls in a full innings

# Regular innings only (super overs already dropped)
base = df[df['innings'].isin([1, 2])]

inn = (base.groupby(['match_id', 'innings'])
       .agg(legal_balls=('valid_ball', 'sum'),
            runs=('ball_runs_total', 'sum'),
            wkts=('player_out', 'count'),        # total dismissals (NaN on non-wicket balls)
            target=('runs_target', 'first'))
       .unstack('innings'))

win_score2 = inn[('target', 2)] if TARGET_IS_WINNING_SCORE else inn[('target', 2)] + 1

all_out1  = inn[('wkts', 1)] >= 10
all_out2  = inn[('wkts', 2)] >= 10
chase_won = inn[('runs', 2)] >= win_score2

no_inn2 = inn[('legal_balls', 2)].isna()                      # match ended after innings 1
short1  = (inn[('legal_balls', 1)] < FULL_BALLS) & ~all_out1
short2  = ((inn[('legal_balls', 2)] < FULL_BALLS) & ~all_out2
           & ~chase_won & ~no_inn2)
revised = (~no_inn2) & (win_score2 != inn[('runs', 1)] + 1)   # DLS / revised target

flag = short1 | short2 | revised | no_inn2

reason = (short1.map({True: 'short1;', False: ''})
          + short2.map({True: 'short2;', False: ''})
          + revised.map({True: 'revised;', False: ''})
          + no_inn2.map({True: 'no_inn2;', False: ''}))

flagged = inn[flag].assign(reason=reason[flag])
print(f"{len(flagged)} flagged of {len(inn)} matches")
print(flagged[[('legal_balls', 1), ('legal_balls', 2),
               ('runs', 1), ('runs', 2),
               ('wkts', 1), ('wkts', 2),
               ('target', 2), ('reason', '')]])

# ---- Hand-review step: edit these after looking at the printout ----
false_positives = []   # match_ids flagged by mistake, to unflag
missed          = []   # match_ids you found by hand that the code missed

rain_ids = (set(flagged.index) - set(false_positives)) | set(missed)

# ---- Create the column (match-level, so both innings rows get the same value) ----
df['rain_affected'] = df['match_id'].isin(rain_ids).astype(int)

# Save the list for your report
pd.Series(sorted(rain_ids), name='match_id').to_csv('rain_affected_matches.csv', index=False)
print(df.groupby('rain_affected')['match_id'].nunique())

50 flagged of 1243 matches
         legal_balls          runs        wkts      target  \
innings            1      2      1      2    1    2      2   
match_id                                                     
335987         119.0  109.0  166.0  168.0  8.0  4.0  167.0   
336010          96.0   96.0  129.0  124.0  7.0  4.0  130.0   
336012         108.0   96.0  122.0  126.0  9.0  1.0  123.0   
336022          66.0   48.0  118.0   94.0  4.0  3.0   89.0   
336025         120.0   48.0  149.0   55.0  5.0  0.0   53.0   
392183          72.0   29.0  104.0   58.0  7.0  0.0   54.0   
392186         120.0   56.0  158.0   79.0  6.0  1.0   69.0   
392197         120.0   92.0  187.0   95.0  6.0  9.0  188.0   
392202         120.0  117.0  164.0  126.0  5.0  9.0  165.0   
392214         108.0  108.0  185.0  174.0  3.0  3.0  187.0   
419114         120.0   99.0  218.0  120.0  7.0  9.0  219.0   
501202         119.0   79.0  112.0  113.0  8.0  3.0  113.0   
501215         102.0   90.0  131.0  135.0  

In [46]:
df.to_csv('prog3.csv',index=False)